# JWT scope와 AgentCore Gateway Interceptors를 활용한 세분화된 액세스 제어

## 개요

이 Notebook에서는 **Gateway interceptors**와 **JWT scope**를 사용하여 **AgentCore Gateway**에 **세분화된 액세스 제어(FGAC)**를 적용하는 방법을 알아봅니다. MCP target에서 제공하는 도구 수와 관계없이 에이전트 엔드포인트를 보호할 수 있는 재사용 가능한 패턴을 익히는 것이 목표입니다.

### 이것이 중요한 이유

에이전트가 확장되면 다음과 같은 기능이 필요할 수 있습니다.

- 특정 사용자가 호출할 수 있는 **도구** 제한  
- **민감한 작업**(cancelOrder, updateOrder, deleteOrder 등)에 대한 액세스 제어  
- 사용자 권한에 따른 **semantic search 결과** 필터링 또는 마스킹  
- 사용자에게 **볼 수 있도록 허용된 도구만 표시**  
- JWT token이 제공하는 범위를 넘어서는 사용자 지정 권한 부여 로직 적용  
- 개별 도구나 Runtime을 수정하지 않고 **중앙 집중식 거버넌스** 적용  

Gateway interceptors를 사용하면 **에이전트, Runtime 또는 MCP server를 수정하지 않고** 확장 가능한 플러그 앤 플레이 방식으로 이러한 제어를 구현할 수 있습니다.  
모든 요청이 자연스럽게 통과하는 **Gateway 수준**에서 정책을 적용합니다.

---

## 이 튜토리얼에서 다루는 내용

다음 세 가지 주요 Gateway 작업에 FGAC를 구현합니다.

1. 🛠️ **FGAC를 적용한 도구 호출(request gateway interceptor)**  
   Gateway interceptor를 사용하여 `tools/call`에 도구별 권한을 적용합니다.  
   ![도구 호출](images/invoke-tool.png)

2. 🔍 **FGAC를 적용한 Semantic Search(response gateway interceptor)**  
   사용자가 허용된 데이터만 볼 수 있도록 Cognito scope를 기준으로 검색 결과를 필터링합니다.  
   ![도구 검색](images/search-tool.png)

3. 📋 **FGAC를 적용한 도구 목록 조회(response gateway interceptor)**  
   사용자가 권한을 가진 도구만 볼 수 있도록 도구 카탈로그를 동적으로 필터링합니다.  
   ![도구 목록](images/list-tool.png)

---

## Gateway Interceptors를 사용하는 이유

Gateway interceptors를 사용하면 다음 작업을 수행할 수 있습니다.

- **세분화된 액세스 제어 구현**  
  사용자별, 도구별, 작업별 권한 부여 규칙을 적용합니다.

- **사용자 지정 권한 부여 로직 삽입**  
  동적 규칙이나 외부 정책을 사용하여 정적 JWT 검증 이상의 제어를 구현합니다.

- **감사 및 거버넌스**  
  도구 사용 시도를 기록하고 규정 준수 현황을 파악할 수 있게 합니다.

- **요청/응답 변환**  
  사용자가 보기 전에 데이터를 마스킹하거나 요청을 수정하고 응답을 필터링합니다.

interceptor는 **Gateway 계층**에 연결되므로 기반이 되는 **모든** MCP server 또는 Runtime에 중앙 정책을 적용할 수 있습니다.

---

## 튜토리얼 세부 정보

| 항목                     | 세부 정보                                                                                       |
|--------------------------|-------------------------------------------------------------------------------------------------|
| **튜토리얼 유형**        | 대화형                                                                                          |
| **AgentCore 구성 요소**  | AgentCore Gateway, Gateway Interceptors                                                         |
| **Gateway Target 유형**  | MCP Server(AgentCore Runtime에서 실행되는 FastMCP)                                              |
| **Interceptor 유형**     | AWS Lambda(request + response)                                                                  |
| **인바운드 인증 IdP**    | Amazon Cognito(CUSTOM\_JWT authorizer)                                                         |
| **액세스 제어**          | Cognito scope + Gateway interceptors를 사용하는 FGAC                                            |
| **튜토리얼 구성 요소**   | Gateway, Runtime MCP Server, Cognito, Gateway Interceptors, MCP tools                            |
| **튜토리얼 분야**        | 여러 분야 공통                                                                                  |
| **예제 난이도**          | 초급~중급                                                                                       |
| **사용 SDK**             | boto3                                                                                            |

---

## 사전 요구 사항

이 튜토리얼을 실행하려면 다음이 필요합니다.

- Jupyter notebook(Python kernel)
- 다음 서비스에 대한 권한이 있는 AWS 자격 증명:
  - Lambda
  - IAM
  - Cognito
  - DynamoDB(여기서는 사용하지 않지만 실제 FGAC에서 유용한 경우가 많음)
  - AgentCore 서비스(컨트롤 플레인 + Runtime)
- Python 3.13 이상
- AWS Lambda, IAM roles, Cognito 및 AgentCore Gateway에 대한 기본 지식

> ⚠️ **참고:** 마지막의 정리 섹션에서는 이 튜토리얼에서 생성한 AWS 리소스(Gateway, Lambda, IAM roles 등)를 삭제합니다. 모든 리소스를 제거할 준비가 되었을 때만 실행하세요.


In [ ]:
# 현재 디렉터리의 requirements 파일에서 패키지 설치
!pip install -r requirements.txt --quiet

In [ ]:
# 필수 라이브러리를 가져오고 리소스 이름에 사용할 고유 타임스탬프 생성

import boto3
import json
import time
import zipfile
import subprocess
import tempfile
import io
import os
import sys
import requests
import uuid
from datetime import datetime

from botocore.exceptions import ClientError

print("✓ Libraries imported")

# 고유한 이름을 위한 타임스탬프 생성
timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
print(f"Using timestamp: {timestamp}")

# MCP Target 이름 정의
gateway_target_name = f"mcp-target-{timestamp}"

# AWS 자격 증명 및 리전 구성
os.environ["AWS_DEFAULT_REGION"] = os.environ.get("AWS_REGION", "us-east-1")
REGION = os.environ["AWS_DEFAULT_REGION"]

print(f"Region: {REGION}")

In [ ]:
# 유틸리티를 가져오고 로깅 구성

# 현재 스크립트의 디렉터리 가져오기
if "__file__" in globals():
    current_dir = os.path.dirname(os.path.abspath(__file__))
else:
    current_dir = os.getcwd()  # __file__이 정의되지 않은 경우의 대체 경로(예: Jupyter)

# utils.py가 있는 디렉터리로 이동(한 단계 위)
utils_dir = os.path.abspath(os.path.join(current_dir, ".."))

# sys.path에 추가
sys.path.insert(0, utils_dir)

# utils 가져오기
import utils

# 로깅 설정
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    handlers=[logging.StreamHandler()],
)

logging.getLogger("strands").setLevel(logging.INFO)

print("✓ Logging configured, utils imported")

## Part 1: Gateway 인바운드 인증을 위한 Cognito User Pool 및 Scope

먼저 FGAC에 사용할 scope를 정의하는 **Cognito User Pool**과 **Resource Server**를 생성합니다.

- MCP target에 대한 **전체 액세스** scope  
- **개별 도구**에 대한 scope: `getOrder`, `updateOrder`, `cancelOrder`, `deleteOrder`

이 scope는 access token에 포함되며 다음 두 곳에서 사용됩니다.

- **request gateway interceptor**(도구 호출용)  
- **response gateway interceptor**(semantic search 및 tools/list용)


In [ ]:
# Gateway 인바운드 권한 부여를 위한 Amazon Cognito Pool 생성

# 실행할 때마다 새로운 pool / resource server / client를 사용하도록 타임스탬프가 포함된 이름 사용
USER_POOL_NAME = f"gateway-interceptor-pool-{timestamp}"
RESOURCE_SERVER_ID = f"gateway-interceptor-id-{timestamp}"
RESOURCE_SERVER_NAME = "gateway-interceptor-name"
CLIENT_NAME = f"gateway-interceptor-client-{timestamp}"

# Scope는 이번 실행의 현재 gateway_target_name을 기준으로 지정
SCOPES = [
    # MCP target에 대한 전체 액세스
    {
        "ScopeName": gateway_target_name,
        "ScopeDescription": "Full access to all tools in MCP target",
    },
    # 특정 도구 액세스
    {
        "ScopeName": f"{gateway_target_name}:getOrder",
        "ScopeDescription": "Access to getOrder tool only",
    },
    {
        "ScopeName": f"{gateway_target_name}:updateOrder",
        "ScopeDescription": "Access to updateOrder tool only",
    },
    {
        "ScopeName": f"{gateway_target_name}:cancelOrder",
        "ScopeDescription": "Access to cancelOrder tool only",
    },
    {
        "ScopeName": f"{gateway_target_name}:deleteOrder",
        "ScopeDescription": "Access to deleteOrder tool only",
    },
]

# Cognito 형식의 전체 scope 문자열: "<resource-server-id>/<scope-name>"
scope_names = [f"{RESOURCE_SERVER_ID}/{scope['ScopeName']}" for scope in SCOPES]
scopeString = " ".join(scope_names)

cognito = boto3.client("cognito-idp", region_name=REGION)

print("Creating or retrieving Cognito resources...")
gw_user_pool_id = utils.get_or_create_user_pool(cognito, USER_POOL_NAME)
print(f"User Pool ID: {gw_user_pool_id}")

utils.get_or_create_resource_server(cognito, gw_user_pool_id, RESOURCE_SERVER_ID, RESOURCE_SERVER_NAME, SCOPES)
print("Resource server ensured.")

gw_client_id, gw_client_secret = utils.get_or_create_m2m_client(
    cognito, gw_user_pool_id, CLIENT_NAME, RESOURCE_SERVER_ID, scope_names
)
print(f"Client ID: {gw_client_id}")

# 이후 Gateway authorizer와 utils.get_token에서 사용할 discovery URL
gw_cognito_discovery_url = (
    f"https://cognito-idp.{REGION}.amazonaws.com/{gw_user_pool_id}/.well-known/openid-configuration"
)
gw_jwks_url = f"https://cognito-idp.{REGION}.amazonaws.com/{gw_user_pool_id}/.well-known/jwks.json"
print(gw_cognito_discovery_url)

## Part 2: Gateway Interceptors용 IAM Role
Gateway Interceptors(request 및 response)가 모두 사용할 단일 IAM role을 생성합니다.

In [ ]:
# Gateway Interceptors용 IAM role 생성

iam_client = boto3.client("iam", region_name=REGION)

lambda_trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "lambda.amazonaws.com"},
            "Action": "sts:AssumeRole",
        }
    ],
}

lambda_role_name = f"GatewayInterceptorRole-{timestamp}"

lambda_role_response = iam_client.create_role(
    RoleName=lambda_role_name,
    AssumeRolePolicyDocument=json.dumps(lambda_trust_policy),
    Description="IAM role for Gateway Interceptors",
)

# 기본 Lambda 실행 정책 연결
iam_client.attach_role_policy(
    RoleName=lambda_role_name,
    PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole",
)

lambda_role_arn = lambda_role_response["Role"]["Arn"]
print(f"Lambda IAM role created: {lambda_role_arn}")

# role을 사용할 수 있을 때까지 대기
print("Waiting for Lambda role to be available...")
time.sleep(10)
print("✓ Lambda role ready")

## Part 3: 도구 호출을 위한 Request Gateway Interceptor

이 interceptor는 다음 작업을 수행합니다.

- Authorization header 검증
- JWT를 디코딩하여 `scope` claim 확인
- MCP `tools/call` 요청에서 도구 이름 추출
- token의 scope가 해당 도구 호출을 허용하는지 확인
- 다음 결과 반환:
  - **허용되지 않은 경우** 403 오류 MCP 응답
  - **허용된 경우** 요청을 그대로 전달

이를 통해 **호출 시점에 FGAC**를 적용합니다.


In [ ]:
# 도구 호출 FGAC를 위한 request gateway interceptor 함수 생성


def create_request_gateway_interceptor():
    """Request gateway interceptor 함수를 생성한다."""
    lambda_client = boto3.client("lambda", region_name=REGION)

    lambda_code = '''import json
import time
import urllib.request
from jose import jwk, jwt
from jose.utils import base64url_decode

# gateway target 이름과 resource server ID 정의
gateway_target_name = "TARGET_GATEWAY_NAME_PLACEHOLDER"
resource_server_id = "RESOURCE_SERVER_ID_PLACEHOLDER"
jwks_url = "JWKS_URL_PLACEHOLDER"
client_id = "CLIENT_ID_PLACEHOLDER"

with urllib.request.urlopen(jwks_url) as f:
    response = f.read()
keys = json.loads(response.decode('utf-8'))['keys']

# Request interceptor에서 FGAC로 차단하지 않을 system / internal tool
# 필요한 경우 response interceptor에서 대신 필터링함
SYSTEM_TOOLS = {
    "x_amz_bedrock_agentcore_search",  # 의미 기반 검색 도구
}

def decode_jwt_payload(token):
    """JWT payload를 검증하고 decode한다."""
    headers = jwt.get_unverified_headers(token)
    kid = headers['kid']
    
    # 일치하는 key 찾기
    key_index = -1
    for i, key in enumerate(keys):
        if kid == key['kid']:
            key_index = i
            break
    
    if key_index == -1:
        raise Exception('Public key not found in jwks.json')
    
    # Public key 구성
    public_key = jwk.construct(keys[key_index])
    
    # Message와 signature 가져오기
    message, encoded_signature = str(token).rsplit('.', 1)
    decoded_signature = base64url_decode(encoded_signature.encode('utf-8'))
    
    # Signature 검증
    if not public_key.verify(message.encode("utf8"), decoded_signature):
        raise Exception('Signature verification failed')
    
    # Claim 가져오기
    claims = jwt.get_unverified_claims(token)
    
    # 만료 여부 확인
    if time.time() > claims['exp']:
        raise Exception('Token is expired')
    
    # client_id 확인
    if claims['client_id'] != client_id:
        raise Exception('Token was not issued for this audience')
    
    # Token 용도 확인
    if claims.get('token_use') != 'access':
        raise Exception("Invalid token use: must be 'access' token")
    
    return claims

def check_tool_authorization(scopes, tool_name):
    """
    scope를 기준으로 사용자에게 특정 도구 권한이 있는지 확인합니다.

    Cognito는 다음 형식으로 scope를 반환합니다.
      "<resource_server_id>/<actual_scope>"
    예:
      "agentcore-gateway-interceptor-id-20251119-141611/mcp-target-20251119-141611:getOrder"

    슬래시 뒤의 부분만 사용합니다.
      "mcp-target-20251119-141611:getOrder"
    """
    if not scopes:
        return False
    
    user_scopes = scopes.split(" ")
    actual_scopes = []
    
    for s in user_scopes:
        # Resource server 접두사가 있으면 제거
        if "/" in s:
            actual_scopes.append(s.split("/", 1)[1])
        else:
            actual_scopes.append(s)

    # 이 MCP 대상의 모든 tool에 대한 전체 접근 권한
    if gateway_target_name in actual_scopes:
        return True
    
    # 특정 tool 권한: <target>:<toolName>
    required_scope = f"{gateway_target_name}:{tool_name}"
    if required_scope in actual_scopes:
        return True
    
    return False

def extract_tool_name(body):
    """MCP tools/call 요청 본문에서 도구 이름을 추출합니다."""
    try:
        if isinstance(body, dict):
            params = body.get("params", {})
            tool_name = params.get("name", "")
            # 도구 이름 형식: <target>___<toolName>
            if "___" in tool_name:
                return tool_name.split("___")[-1]
            return tool_name
    except Exception:
        pass
    return None

def build_pass_through_response(auth_header, body):
    """요청이 target에 도달하도록 통과 응답을 반환합니다."""
    return {
        "interceptorOutputVersion": "1.0",
        "mcp": {
            "transformedGatewayRequest": {
                "headers": {
                    "Authorization": auth_header,
                    "Content-Type": "application/json"
                },
                "body": body
            }
        }
    }

def build_error_response(message, body, status_code=403):
    """MCP 형식의 오류 응답을 반환합니다."""
    return {
        "interceptorOutputVersion": "1.0",
        "mcp": {
            "transformedGatewayResponse": {
                "statusCode": status_code,
                "body": {
                    "jsonrpc": "2.0",
                    "id": body.get("id", "unknown") if isinstance(body, dict) else "unknown",
                    "error": {
                        "code": -32600,
                        "message": message
                    }
                }
            }
        }
    }

def lambda_handler(event, context):
    print(f"Received event: {json.dumps(event)}")
    
    # 올바른 구조에서 gateway request 추출
    mcp_data = event.get("mcp", {})
    gateway_request = mcp_data.get("gatewayRequest", {})
    headers = gateway_request.get("headers", {})
    body = gateway_request.get("body", {})
    
    # Authorization header 추출
    auth_header = headers.get("Authorization", "")
    
    # 모든 request에 Bearer token이 있도록 강제
    if not auth_header.startswith("Bearer "):
        response = build_error_response("No authorization token provided", body)
        print(f"Returning error response (no token): {json.dumps(response)}")
        return response
    
    # Token과 scope decode
    try:
        token = auth_header.replace("Bearer ", "")
        decoded_token = decode_jwt_payload(token)
        scopes = decoded_token.get("scope", "")
        
        method = body.get("method", "")
        tool_name = extract_tool_name(body)
        
        print(f"Decoded scopes (raw): {scopes}")
        print(f"MCP method: {method}")
        print(f"Requested tool: {tool_name}")
        
        # 1) tools/list 허용: 표시할 tool에 대한 FGAC는 반환 목록을 필터링하는
        #    response gateway interceptor에서 적용함
        if method == "tools/list":
            print("tools/list request detected - skipping tool-level FGAC in request gateway interceptor")
            response = build_pass_through_response(auth_header, body)
            print(f"Returning pass-through response for tools/list: {json.dumps(response)}")
            return response
        
        # 2) System tool(예: x_amz_bedrock_agentcore_search)이 통과하도록 허용
        #    Response gateway interceptor가 이후 tool 목록 / 검색 결과를 필터링함
        if tool_name in SYSTEM_TOOLS:
            print(f"System tool '{tool_name}' detected - skipping tool-level FGAC in request gateway interceptor")
            response = build_pass_through_response(auth_header, body)
            print(f"Returning pass-through response for system tool: {json.dumps(response)}")
            return response
        
        # 3) 그 외 모든 tool(getOrder, updateOrder 등의 business tool)에는
        #    scope 기반 FGAC를 적용함
        if not tool_name:
            response = build_error_response("No tool name provided in request", body)
            print(f"Returning error response (no tool name): {json.dumps(response)}")
            return response
        
        if not check_tool_authorization(scopes, tool_name):
            response = build_error_response(
                f"Insufficient permission for tool: {tool_name}",
                body
            )
            print(f"Returning error response (FGAC deny): {json.dumps(response)}")
            return response
    
    except Exception as e:
        print(f"Error while validating token/scopes: {e}")
        response = build_error_response(f"Invalid token - {e}", body)
        print(f"Returning error response (exception): {json.dumps(response)}")
        return response
    
    # 승인됨 → 통과
    response = build_pass_through_response(auth_header, body)
    print(f"Returning pass-through response (authorized): {json.dumps(response)}")
    return response
'''
    # placeholder를 실제 값으로 교체
    lambda_code = lambda_code.replace("TARGET_GATEWAY_NAME_PLACEHOLDER", gateway_target_name)
    lambda_code = lambda_code.replace("RESOURCE_SERVER_ID_PLACEHOLDER", RESOURCE_SERVER_ID)
    lambda_code = lambda_code.replace("JWKS_URL_PLACEHOLDER", gw_jwks_url)
    lambda_code = lambda_code.replace("CLIENT_ID_PLACEHOLDER", gw_client_id)

    # 종속 항목을 포함한 Lambda용 ZIP 파일 생성
    with tempfile.TemporaryDirectory() as temp_dir:
        # 종속 항목 설치
        subprocess.run(
            ["pip", "install", "-r", "requirements_lambda.txt", "-t", temp_dir],
            check=True,
        )

        # zip buffer 생성
        zip_buffer = io.BytesIO()
        with zipfile.ZipFile(zip_buffer, "w", zipfile.ZIP_DEFLATED) as zip_file:
            # Lambda 함수 코드 추가
            zip_file.writestr("lambda_function.py", lambda_code)

            # 모든 종속 파일 추가
            for root, dirs, files in os.walk(temp_dir):
                for file in files:
                    file_path = os.path.join(root, file)
                    arc_name = os.path.relpath(file_path, temp_dir)
                    zip_file.write(file_path, arc_name)

    zip_buffer.seek(0)

    # Lambda 함수 생성
    lambda_function_name = f"request-gateway-interceptor-{timestamp}"

    lambda_response = lambda_client.create_function(
        FunctionName=lambda_function_name,
        Runtime="python3.13",
        Role=lambda_role_arn,
        Handler="lambda_function.lambda_handler",
        Code={"ZipFile": zip_buffer.read()},
        Description="Request Gateway Interceptor for AgentCore Gateway - Tool Invocation FGAC",
    )

    request_lambda_arn = lambda_response["FunctionArn"]
    print(f"Request Gateway Interceptor function created: {request_lambda_arn}")

    return request_lambda_arn


request_lambda_arn = create_request_gateway_interceptor()
print(f"\n✅ Request Gateway interceptor creation completed: {request_lambda_arn}")

## Part 4: Semantic Search 및 Tools/List를 위한 Response Gateway Interceptor

이 interceptor는 다음 작업을 수행합니다.

- 응답의 Authorization header에서 JWT scope 디코딩
- 허용된 scope를 기준으로 도구 목록 필터링
- 다음 항목에 적용:
  - Semantic search 결과(`x_amz_bedrock_agentcore_search`)
  - MCP `tools/list` 응답(도구 목록만 열거)

다음 로직을 사용합니다.

- scope가 MCP target 이름과 같으면 → 해당 target의 모든 도구에 전체 액세스  
- scope가 `mcpTarget:toolName`과 같으면 → 해당 도구에만 액세스  
- `target___toolName` 명명 규칙을 따르지 않는 내부/system 도구는 건너뜀


In [ ]:
def create_response_gateway_interceptor():
    """Response gateway interceptor 함수를 생성한다."""
    lambda_client = boto3.client("lambda", region_name=REGION)

    lambda_code = '''import json
import time
import urllib.request
from jose import jwk, jwt
from jose.utils import base64url_decode

# gateway target 이름과 resource server ID 정의
jwks_url = "JWKS_URL_PLACEHOLDER"
client_id = "CLIENT_ID_PLACEHOLDER"

with urllib.request.urlopen(jwks_url) as f:
    response = f.read()
keys = json.loads(response.decode('utf-8'))['keys']

def decode_jwt_payload(token):
    """JWT payload를 검증하고 decode한다."""
    headers = jwt.get_unverified_headers(token)
    kid = headers['kid']
    
    # 일치하는 key 찾기
    key_index = -1
    for i, key in enumerate(keys):
        if kid == key['kid']:
            key_index = i
            break
    
    if key_index == -1:
        raise Exception('Public key not found in jwks.json')
    
    # Public key 구성
    public_key = jwk.construct(keys[key_index])
    
    # Message와 signature 가져오기
    message, encoded_signature = str(token).rsplit('.', 1)
    decoded_signature = base64url_decode(encoded_signature.encode('utf-8'))
    
    # Signature 검증
    if not public_key.verify(message.encode("utf8"), decoded_signature):
        raise Exception('Signature verification failed')
    
    # Claim 가져오기
    claims = jwt.get_unverified_claims(token)
    
    # 만료 여부 확인
    if time.time() > claims['exp']:
        raise Exception('Token is expired')
    
    # client_id 확인
    if claims['client_id'] != client_id:
        raise Exception('Token was not issued for this audience')
    
    # Token 용도 확인
    if claims.get('token_use') != 'access':
        raise Exception("Invalid token use: must be 'access' token")
    
    return claims

def filter_tools_by_scope(tools, allowed_scopes):
    """사용자 지정 scope에 따라 tool을 필터링한다."""
    if not allowed_scopes:
        return []
    
    filtered_tools = []
    for tool in tools:
        tool_name = tool.get("name", "")
        
        # Target separator가 없는 system 생성 MCP tool은 건너뜀
        if "___" not in tool_name:
            continue
            
        mcp_target = tool_name.split("___")[0]
        tool_action = tool_name.split("___")[1]
        
        for scope in allowed_scopes:
            # 실제 scope를 얻도록 resource server 접두사 제거
            actual_scope = scope.split("/")[-1] if "/" in scope else scope
            
            # MCP 대상에 대한 전체 접근 권한
            if actual_scope == mcp_target:
                filtered_tools.append(tool)
                break
            # 특정 tool 접근 권한
            elif actual_scope == f"{mcp_target}:{tool_action}":
                filtered_tools.append(tool)
                break
    
    return filtered_tools

def lambda_handler(event, context):
    print(f"Received event: {json.dumps(event)}")
    
    # Gateway response 추출
    mcp_data = event.get("mcp", {})
    gateway_response = mcp_data.get("gatewayResponse", {})
    headers = gateway_response.get("headers", {})
    body = gateway_response.get("body", {})

    # Authorization header를 그대로 전달
    auth_header = headers.get("Authorization", "")
    token = auth_header.replace("Bearer ", "") if auth_header.startswith("Bearer ") else ""

    try:
        # 검증 없이 JWT payload 파싱
        claims = decode_jwt_payload(token)
        
        # Claim에서 scope 추출
        scope_string = claims.get("scope", "")
        scopes = scope_string.split() if scope_string else []
        print(f"Extracted scopes: {scopes}")
        
        # Gateway response에서 tool 가져오기(semantic search 및 tools/list 지원)
        result = body.get("result", {})
        tools = result.get("tools", [])
        if not tools:
            structured_content = result.get("structuredContent", {})
            tools = structured_content.get("tools", [])
        print(f"Available tools: {[tool.get('name') for tool in tools]}")
        
        # Scope에 따라 tool 필터링
        filtered_tools = filter_tools_by_scope(tools, scopes)
        print(f"Filtered tools: {[tool.get('name') for tool in filtered_tools]}")
        
        # 필터링된 tool로 body 업데이트
        filtered_body = body.copy()
        if "result" in filtered_body:
            if "structuredContent" in filtered_body["result"]:
                filtered_body["result"]["structuredContent"]["tools"] = filtered_tools
            else:
                filtered_body["result"]["tools"] = filtered_tools
            
            # Content array가 있고 text에 tool JSON이 포함되어 있으면 필터링
            if "content" in filtered_body["result"]:
                for content_item in filtered_body["result"]["content"]:
                    if content_item.get("type") == "text" and "text" in content_item:
                        try:
                            content_data = json.loads(content_item["text"])
                            if "tools" in content_data:
                                content_filtered_tools = filter_tools_by_scope(content_data["tools"], scopes)
                                content_data["tools"] = content_filtered_tools
                                content_item["text"] = json.dumps(content_data)
                        except (json.JSONDecodeError, KeyError):
                            pass
    except Exception as e:
        print(f"Error processing JWT or filtering tools: {e}")
        filtered_body = body
    else:
        # Except가 발생하지 않았을 때만 filtered_body 설정
        pass

    # try/except에서 filtered_body가 설정되지 않았으면 원본 body 사용
    if "filtered_body" not in locals():
        filtered_body = body

    # 변환된 응답 반환
    response = {
        "interceptorOutputVersion": "1.0",
        "mcp": {
            "transformedGatewayResponse" : {
                "statusCode": 200,
                "headers": {
                    "Accept": "application/json",
                    "Authorization": auth_header
                },
                "body": filtered_body
            }
        }   
    }
    
    print(f"Returning response: {json.dumps(response)}")
    return response
'''

    # placeholder를 실제 값으로 교체
    lambda_code = lambda_code.replace("JWKS_URL_PLACEHOLDER", gw_jwks_url)
    lambda_code = lambda_code.replace("CLIENT_ID_PLACEHOLDER", gw_client_id)

    # 종속 항목을 포함한 Lambda용 ZIP 파일 생성
    with tempfile.TemporaryDirectory() as temp_dir:
        # 종속 항목 설치
        subprocess.run(
            ["pip", "install", "-r", "requirements_lambda.txt", "-t", temp_dir],
            check=True,
        )

        # zip buffer 생성
        zip_buffer = io.BytesIO()
        with zipfile.ZipFile(zip_buffer, "w", zipfile.ZIP_DEFLATED) as zip_file:
            # Lambda 함수 코드 추가
            zip_file.writestr("lambda_function.py", lambda_code)

            # 모든 종속 파일 추가
            for root, dirs, files in os.walk(temp_dir):
                for file in files:
                    file_path = os.path.join(root, file)
                    arc_name = os.path.relpath(file_path, temp_dir)
                    zip_file.write(file_path, arc_name)

    zip_buffer.seek(0)

    # Lambda 함수 생성
    lambda_function_name = f"response-gateway-interceptor-{timestamp}"

    lambda_response = lambda_client.create_function(
        FunctionName=lambda_function_name,
        Runtime="python3.13",
        Role=lambda_role_arn,
        Handler="lambda_function.lambda_handler",
        Code={"ZipFile": zip_buffer.read()},
        Description="Response Gateway Interceptor for AgentCore Gateway - Search & List FGAC",
    )

    response_lambda_arn = lambda_response["FunctionArn"]
    print(f"Response Gateway Interceptor function created: {response_lambda_arn}")

    return response_lambda_arn


response_lambda_arn = create_response_gateway_interceptor()
print(f"\n✅ Response Gateway interceptor creation completed: {response_lambda_arn}")

## Part 5: 두 Interceptor를 사용하는 AgentCore Gateway 생성

이제 다음과 같이 AgentCore Gateway를 생성합니다.

- **프로토콜:** MCP  
- **검색 유형:** SEMANTIC  
- **Request Gateway interceptor:** 도구 호출 FGAC  
- **Response Gateway interceptor:** Semantic search 및 tools/list FGAC  
- **Authorizer:** 위에서 생성한 Cognito User Pool과 client를 사용하는 CUSTOM\_JWT


In [ ]:
gateway_role_name = f"BedrockAgentCoreGatewayRole-{timestamp}"
agentcore_gateway_iam_role = utils.create_agentcore_gateway_role(gateway_role_name)
role_arn = agentcore_gateway_iam_role["Role"]["Arn"]
print("Agentcore gateway role ARN: ", role_arn)

In [ ]:
# boto3 client를 사용하여 interceptor가 적용된 Gateway 생성
def create_gateway_with_interceptors():
    gateway_client = boto3.client("bedrock-agentcore-control", region_name=REGION)

    print("Creating gateway with interceptors...")
    gateway_response = gateway_client.create_gateway(
        name=f"gateway-interceptor-{timestamp}",
        roleArn=role_arn,
        protocolType="MCP",
        protocolConfiguration={"mcp": {"supportedVersions": ["2025-03-26"], "searchType": "SEMANTIC"}},
        interceptorConfigurations=[
            {
                "interceptor": {"lambda": {"arn": request_lambda_arn}},
                "interceptionPoints": ["REQUEST"],
                "inputConfiguration": {"passRequestHeaders": True},
            },
            {
                "interceptor": {"lambda": {"arn": response_lambda_arn}},
                "interceptionPoints": ["RESPONSE"],
                "inputConfiguration": {"passRequestHeaders": False},
            },
        ],
        authorizerType="CUSTOM_JWT",
        authorizerConfiguration={
            "customJWTAuthorizer": {
                "discoveryUrl": gw_cognito_discovery_url,
                "allowedClients": [gw_client_id],
            }
        },
    )

    print("Gateway create response:", gateway_response)

    gateway_id = gateway_response["gatewayId"]
    gateway_url = gateway_response["gatewayUrl"]
    print(f"Gateway with interceptors created: {gateway_id}")

    # Gateway가 준비될 때까지 대기
    print("Waiting for gateway to be ready...")
    while True:
        status_response = gateway_client.get_gateway(gatewayIdentifier=gateway_id)
        current_status = status_response.get("status", "UNKNOWN")
        print(f"Gateway status: {current_status}")
        if current_status == "READY":
            print(f"Final gateway details: {json.dumps(status_response, indent=2, default=str)}")
            break
        time.sleep(10)

    print("Gateway is now ready")
    return gateway_id, gateway_url


gateway_id, gateway_url = create_gateway_with_interceptors()
print(f"\n✅ Gateway creation completed: Gateway Id {gateway_id}")
print(f"Gateway Url: {gateway_url}")

## Part 6: 샘플 MCP Server를 생성하고 AgentCore Runtime에 호스팅

이제 다음 작업을 수행합니다.

1. **Runtime용 Cognito User Pool**을 생성합니다(Gateway에서 Runtime으로의 아웃바운드 인증용).
2. 다음 4개 도구가 포함된 간단한 **FastMCP server**를 AgentCore Runtime에 호스팅합니다.
   - `getOrder`
   - `updateOrder`
   - `cancelOrder`
   - `deleteOrder`
3. AgentCore Runtime에 호스팅합니다.


In [ ]:
# Runtime용 Cognito User Pool 생성(Gateway의 아웃바운드 인증)

RUNTIME_USER_POOL_NAME = f"gateway-interceptor-rt-pool-{timestamp}"
RUNTIME_RESOURCE_SERVER_ID = f"gateway-interceptor-rt-id-{timestamp}"
RUNTIME_RESOURCE_SERVER_NAME = "gateway-interceptor-rt-name"
RUNTIME_CLIENT_NAME = f"gateway-interceptor-runtime-rt-{timestamp}"

RUNTIME_SCOPES = [
    {
        "ScopeName": "tools",
        "ScopeDescription": "Scope for search,list and invoke the agentcore gateway",
    },
]

runtime_scope_names = [f"{RUNTIME_RESOURCE_SERVER_ID}/{scope['ScopeName']}" for scope in RUNTIME_SCOPES]
runtimeScopeString = " ".join(runtime_scope_names)

cognito = boto3.client("cognito-idp", region_name=REGION)

print("Creating or retrieving Cognito resources for Runtime...")
runtime_user_pool_id = utils.get_or_create_user_pool(cognito, RUNTIME_USER_POOL_NAME)
print(f"Runtime User Pool ID: {runtime_user_pool_id}")

utils.get_or_create_resource_server(
    cognito,
    runtime_user_pool_id,
    RUNTIME_RESOURCE_SERVER_ID,
    RUNTIME_RESOURCE_SERVER_NAME,
    RUNTIME_SCOPES,
)
print("Runtime resource server ensured.")

runtime_client_id, runtime_client_secret = utils.get_or_create_m2m_client(
    cognito,
    runtime_user_pool_id,
    RUNTIME_CLIENT_NAME,
    RUNTIME_RESOURCE_SERVER_ID,
    runtime_scope_names,
)

print(f"Runtime Client ID: {runtime_client_id}")

runtime_cognito_discovery_url = (
    f"https://cognito-idp.{REGION}.amazonaws.com/{runtime_user_pool_id}/.well-known/openid-configuration"
)
print("Runtime Cognito discovery URL:", runtime_cognito_discovery_url)

In [ ]:
# 샘플 MCP server 파일(FastMCP)을 생성하고 AgentCore Runtime에 호스팅

content = """
from mcp.server.fastmcp import FastMCP

mcp = FastMCP(host="0.0.0.0", stateless_http=True)

@mcp.tool()
def getOrder() -> int:
    '''Get an order'''
    return 123

@mcp.tool()
def updateOrder(orderId: int) -> int:
    '''Update existing order'''
    return 456

@mcp.tool()
def cancelOrder(orderId: int) -> int:
    '''Cancel existing order'''
    return 789

@mcp.tool()
def deleteOrder(orderId: int) -> int:
    '''Delete existing order'''
    return 101

if __name__ == "__main__":
    mcp.run(transport="streamable-http")
"""

with open("mcp_server.py", "w") as f:
    f.write(content)

In [ ]:
# MCP Server를 구성하고 AgentCore Runtime에 배포

from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session

boto_session = Session()
print(f"Using AWS region: {REGION}")

required_files = ["mcp_server.py", "requirements.txt"]
for file in required_files:
    if not os.path.exists(file):
        raise FileNotFoundError(f"Required file {file} not found")
print("All required files found ✓")

agentcore_runtime = Runtime()

auth_config = {
    "customJWTAuthorizer": {
        "allowedClients": [runtime_client_id],
        "discoveryUrl": runtime_cognito_discovery_url,
    }
}

print("Configuring AgentCore Runtime...")
runtime_response = agentcore_runtime.configure(
    entrypoint="mcp_server.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=REGION,
    authorizer_configuration=auth_config,
    protocol="MCP",
    agent_name="ac_gateway_mcp_server",
)
print("Configuration completed ✓")

In [ ]:
# AgentCore Runtime에 MCP Server 시작

print("Launching MCP server to AgentCore Runtime...")
launch_result = agentcore_runtime.launch()

runtime_agent_arn = launch_result.agent_arn
runtime_agent_id = launch_result.agent_id

encoded_arn = runtime_agent_arn.replace(":", "%3A").replace("/", "%2F")

agent_url = f"https://bedrock-agentcore.{REGION}.amazonaws.com/runtimes/{encoded_arn}/invocations?qualifier=DEFAULT"
print("Launch completed ✓")
print(f"Agent ARN: {runtime_agent_arn}")
print(f"Agent ID: {runtime_agent_id}")
print(f"Runtime MCP URL: {agent_url}")

In [ ]:
# MCP Server 인증을 위한 OAuth2 Credential Provider 생성(Gateway → Runtime)

identity_client = boto3.client("bedrock-agentcore-control", region_name=REGION)

cognito_provider = identity_client.create_oauth2_credential_provider(
    name=f"gateway-mcp-server-identity-{timestamp}",
    credentialProviderVendor="CustomOauth2",
    oauth2ProviderConfigInput={
        "customOauth2ProviderConfig": {
            "oauthDiscovery": {
                "discoveryUrl": runtime_cognito_discovery_url,
            },
            "clientId": runtime_client_id,
            "clientSecret": runtime_client_secret,
        }
    },
)
cognito_provider_arn = cognito_provider["credentialProviderArn"]
print("Outbound OAuth2 Credential Provider ARN:", cognito_provider_arn)

## Part 7: Gateway Target 생성 및 MCP Server 등록

이제 AgentCore Gateway Target을 생성하고 MCP Server를 Gateway 뒤의 target으로 등록합니다.

In [ ]:
# MCP server를 가리키는 Gateway Target 생성


def create_gateway_target(gateway_id):
    """gateway target을 생성하고 준비될 때까지 기다립니다."""
    gateway_client = boto3.client("bedrock-agentcore-control", region_name=REGION)

    print("Creating MCP target...")
    target_response = gateway_client.create_gateway_target(
        name=gateway_target_name,
        gatewayIdentifier=gateway_id,
        targetConfiguration={"mcp": {"mcpServer": {"endpoint": agent_url}}},
        credentialProviderConfigurations=[
            {
                "credentialProviderType": "OAUTH",
                "credentialProvider": {
                    "oauthCredentialProvider": {
                        "providerArn": cognito_provider_arn,
                        "scopes": [runtimeScopeString],
                    }
                },
            }
        ],
    )

    target_id = target_response["targetId"]
    print(f"Gateway target created: {target_id}")

    # target이 준비될 때까지 대기
    print("Waiting for target to be ready...")
    while True:
        status_response = gateway_client.get_gateway_target(gatewayIdentifier=gateway_id, targetId=target_id)
        current_status = status_response.get("status", "UNKNOWN")
        print(f"Target status: {current_status}")
        if current_status == "READY":
            print(f"Target status response: {json.dumps(status_response, indent=2, default=str)}")
            break
        elif current_status == "FAILED":
            print("Target creation failed!")
            print(f"Failed target details: {json.dumps(status_response, indent=2, default=str)}")
            raise Exception("Target failed")
        time.sleep(10)

    print("Target is now ready")
    return target_id


target_id = create_gateway_target(gateway_id)
print(f"\n✅ Target creation completed: {target_id}")

## Part 8: FGAC 테스트

이제 다음 세 가지 시나리오를 테스트합니다.

1. **FGAC를 적용한 도구 호출(request gateway interceptor)**  
2. **FGAC를 적용한 Semantic Search(response gateway interceptor)**  
3. **FGAC를 적용한 도구 목록 조회(response gateway interceptor)**

동일한 Cognito User Pool과 client를 사용하되, 서로 다른 scope를 지정하여 token을 요청합니다.

- MCP target의 모든 도구에 대한 전체 액세스  
- 단일 도구에 대한 제한된 액세스(예: `getOrder`)


In [ ]:
def invoke_tool(gateway_url, access_token, tool_name, arguments=None):
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {access_token}",
    }

    # 기본 인수
    if arguments is None:
        arguments = {"orderId": 123} if tool_name != "getOrder" else {}

    payload = {
        "jsonrpc": "2.0",
        "id": "invoke-tool-request",
        "method": "tools/call",
        "params": {
            "name": f"{gateway_target_name}___{tool_name}",
            "arguments": arguments,
        },
    }

    response = requests.post(gateway_url, headers=headers, json=payload)
    return response.json()


def get_access_token_for_scope(scope_label, scope):
    """
    지정한 scope의 access token을 가져오는 헬퍼입니다.
    Cognito가 오류를 반환해 access_token이 없으면 원본 응답을 출력하고,
    KeyError 대신 명확한 예외를 발생시킵니다.
    """
    print(f"\n[Token] Requesting token for {scope_label}")
    print(f"[Token] Using scope: {scope}")

    token_response = utils.get_token(gw_user_pool_id, gw_client_id, gw_client_secret, scope, REGION)
    print(f"[Token] Raw token response:\n{json.dumps(token_response, indent=2)}")

    access_token = token_response.get("access_token")
    if not access_token:
        raise RuntimeError(
            f"Failed to obtain access token for '{scope_label}'. "
            f"Response did not contain 'access_token'. See [Token] output above."
        )
    return access_token


# 1) getOrder scope로 getOrder 호출 → 허용
print("Test 1: getOrder with getOrder scope - SHOULD ALLOW")
scope = f"{RESOURCE_SERVER_ID}/{gateway_target_name}:getOrder"
token = get_access_token_for_scope("getOrder scope", scope)
result = invoke_tool(gateway_url, token, "getOrder")
print(json.dumps(result, indent=2))

# 2) getOrder scope로 updateOrder 호출 → 거부
print("\nTest 2: updateOrder with getOrder scope - SHOULD DENY")
result = invoke_tool(gateway_url, token, "updateOrder")
print(json.dumps(result, indent=2))

# 3) deleteOrder scope로 deleteOrder 호출 → 허용
print("\nTest 3: deleteOrder with deleteOrder scope - SHOULD ALLOW")
scope = f"{RESOURCE_SERVER_ID}/{gateway_target_name}:deleteOrder"
token = get_access_token_for_scope("deleteOrder scope", scope)
result = invoke_tool(gateway_url, token, "deleteOrder")
print(json.dumps(result, indent=2))

# 4) deleteOrder scope로 getOrder 호출 → 거부
print("\nTest 4: getOrder with deleteOrder scope - SHOULD DENY")
result = invoke_tool(gateway_url, token, "getOrder")
print(json.dumps(result, indent=2))

# 5) 전체 액세스(Scope = MCP target) → 모두 허용
print("\nTest 5: All tools with full access scope - SHOULD ALLOW ALL")
scope = f"{RESOURCE_SERVER_ID}/{gateway_target_name}"
token = get_access_token_for_scope("full access scope", scope)

for tool in ["getOrder", "updateOrder", "cancelOrder", "deleteOrder"]:
    print(f"\n  {tool}:")
    result = invoke_tool(gateway_url, token, tool)
    print(json.dumps(result, indent=4))

In [ ]:
def semantic_search(gateway_url, access_token, query):
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {access_token}",
    }

    payload = {
        "jsonrpc": "2.0",
        "id": "semantic-search-request",
        "method": "tools/call",
        "params": {
            "name": "x_amz_bedrock_agentcore_search",
            "arguments": {"query": query},
        },
    }

    response = requests.post(gateway_url, headers=headers, json=payload)
    return response.json()


search_query = "Find me the tools to help cancel and delete my orders"

# 전체 액세스 scope
full_scope = f"{RESOURCE_SERVER_ID}/{gateway_target_name}"
full_token_response = utils.get_token(gw_user_pool_id, gw_client_id, gw_client_secret, full_scope, REGION)
full_token = full_token_response["access_token"]

print("\n=== Semantic Search with FULL ACCESS scope ===")
full_search_results = semantic_search(gateway_url, full_token, search_query)
print(json.dumps(full_search_results, indent=2))

# 제한된 scope(getOrder만 허용)
limited_scope = f"{RESOURCE_SERVER_ID}/{gateway_target_name}:getOrder"
limited_token_response = utils.get_token(gw_user_pool_id, gw_client_id, gw_client_secret, limited_scope, REGION)
limited_token = limited_token_response["access_token"]

print("\n=== Semantic Search with LIMITED scope (getOrder only) ===")
limited_search_results = semantic_search(gateway_url, limited_token, search_query)
print(json.dumps(limited_search_results, indent=2))

# 결과 비교
print("\n=== SEMANTIC SEARCH TOOL COMPARISON ===")
full_tools = full_search_results.get("result", {}).get("structuredContent", {}).get("tools", [])
limited_tools = limited_search_results.get("result", {}).get("structuredContent", {}).get("tools", [])

print(f"Full access tools count: {len(full_tools)}")
print(f"Limited access tools count: {len(limited_tools)}")
print(f"Full access tools: {[tool.get('name') for tool in full_tools]}")
print(f"Limited access tools: {[tool.get('name') for tool in limited_tools]}")

In [ ]:
def list_tools(gateway_url, access_token):
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {access_token}",
    }

    payload = {
        "jsonrpc": "2.0",
        "id": str(uuid.uuid4()),
        "method": "tools/list",
        "params": {},  # MCP에서는 비어 있더라도 params가 있어야 함
    }

    response = requests.post(gateway_url, headers=headers, json=payload)
    return response.json()


# 전체 액세스 scope
full_scope = f"{RESOURCE_SERVER_ID}/{gateway_target_name}"
full_token_response = utils.get_token(gw_user_pool_id, gw_client_id, gw_client_secret, full_scope, REGION)
full_token = full_token_response["access_token"]

print("\n=== MCP tools/list with FULL ACCESS scope ===")
full_list_results = list_tools(gateway_url, full_token)
print(json.dumps(full_list_results, indent=2))

# 제한된 scope(getOrder만 허용)
limited_scope = f"{RESOURCE_SERVER_ID}/{gateway_target_name}:getOrder"
limited_token_response = utils.get_token(gw_user_pool_id, gw_client_id, gw_client_secret, limited_scope, REGION)
limited_token = limited_token_response["access_token"]

print("\n=== MCP tools/list with LIMITED scope (getOrder only) ===")
limited_list_results = list_tools(gateway_url, limited_token)
print(json.dumps(limited_list_results, indent=2))

# 결과 비교
print("\n=== TOOLS/LIST FGAC COMPARISON ===")
full_list_tools = full_list_results.get("result", {}).get("tools", [])
limited_list_tools = limited_list_results.get("result", {}).get("tools", [])

print(f"Full access tools count: {len(full_list_tools)}")
print(f"Limited access tools count: {len(limited_list_tools)}")
print(f"Full access tools: {[tool.get('name') for tool in full_list_tools]}")
print(f"Limited access tools: {[tool.get('name') for tool in limited_list_tools]}")

## Part 9: 정리 - 모든 리소스 삭제

> ⚠️ **경고:** 이 섹션에서는 이 Notebook에서 생성한 다음 리소스를 **삭제**합니다.
> - AgentCore Gateway 및 MCP target  
> - Gateway Interceptor 함수  
> - Lambda 및 Gateway용 IAM roles  
> - Runtime MCP server  
> - Cognito 리소스는 자동으로 삭제되지 **않음**(수동 정리 권장)


In [ ]:
# Gateway target 및 Gateway 삭제

print("\nDeleting Gateway target and Gateway...")
gateway_client = boto3.client("bedrock-agentcore-control", region_name=REGION)
utils.delete_gateway(gateway_client, gateway_id)

In [ ]:
# Lambda 함수 삭제(request + response interceptors)

print("\nDeleting Lambda functions...")

lambda_client = boto3.client("lambda", region_name=REGION)

# Request interceptor 삭제
try:
    request_lambda_name = request_lambda_arn.split(":function:")[-1]
    lambda_client.delete_function(FunctionName=request_lambda_name)
    print(f"  ✓ Request interceptor deleted: {request_lambda_name}")
except Exception as e:
    print(f"  ⚠ Error deleting request interceptor: {e}")

# Response interceptor 삭제
try:
    response_lambda_name = response_lambda_arn.split(":function:")[-1]
    lambda_client.delete_function(FunctionName=response_lambda_name)
    print(f"  ✓ Response interceptor deleted: {response_lambda_name}")
except Exception as e:
    print(f"  ⚠ Error deleting response Lambda: {e}")

In [ ]:
# IAM roles 삭제(Lambda + Gateway)

print("\nDeleting IAM roles...")

# Gateway interceptor role의 정책을 분리하고 role 삭제
try:
    print("  Deleting gateway interceptor role...")

    iam_client.detach_role_policy(
        RoleName=lambda_role_name,
        PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole",
    )
    iam_client.delete_role(RoleName=lambda_role_name)
    print(f"    ✓ Gateway interceptor role deleted: {lambda_role_name}")
except ClientError as e:
    if e.response["Error"]["Code"] == "NoSuchEntity":
        print(f"    ⚠ Role not found: {lambda_role_name}")
    else:
        print(f"    ⚠ Error deleting gateway interceptor role: {e}")

# Gateway role의 정책을 분리하고 role 삭제
try:
    print("  Deleting Gateway role...")

    iam_client.detach_role_policy(
        RoleName=gateway_role_name,
        PolicyArn="arn:aws:iam::aws:policy/AdministratorAccess",
    )
    iam_client.delete_role(RoleName=gateway_role_name)
    print(f"    ✓ Gateway role deleted: {gateway_role_name}")
except ClientError as e:
    if e.response["Error"]["Code"] == "NoSuchEntity":
        print(f"    ⚠ Role not found: {gateway_role_name}")
    else:
        print(f"    ⚠ Error deleting Gateway role: {e}")

In [ ]:
# MCP Server 삭제

print("\nDeleting MCP Server...")

runtime_client = boto3.client("bedrock-agentcore-control", region_name=REGION)
runtime_client.delete_agent_runtime(agentRuntimeId=runtime_agent_id)

In [ ]:
# Identity Provider 삭제

print("\nDeleting Identity Provider...")

identity_client.delete_oauth2_credential_provider(name=f"gateway-mcp-server-identity-{timestamp}")

In [ ]:
print("\n" + "=" * 80)
print("  CLEANUP COMPLETE")
print("=" * 80)

print("\n✓ Deleted Resources:")
print(f"  • Gateway: {gateway_id}")
print(f"  • Gateway Target: {target_id}")
print(f"  • Request Interceptor: {request_lambda_arn}")
print(f"  • Response Interceptor: {response_lambda_arn}")
print(f"  • Lambda IAM Role: {lambda_role_name}")
print(f"  • Gateway IAM Role: {gateway_role_name}")
print(f"  • MCP Server: {runtime_agent_id}")

print("\n⚠️ Cognito User Pools are NOT deleted in this script.")
print("   You can remove them manually from the AWS Console if desired.")

print(f"\n📝 Deployment timestamp: {timestamp}")